# Modulo Catalogo — Ricerca base

Prime funzioni della pagina catalogo. Tre operazioni:

1. **Ricerca per titolo** (prefisso, sfrutta l'indice `idx_titolo`)
2. **Scheda film per id** (lettura per `_id`, località del dato)
3. **Filtro per genere** (sfrutta l'indice `idx_genere_rating`)

Ogni funzione ha accanto la query MongoDB che la giustifica. Prerequisiti: `connessione.py` nella stessa cartella e indici già creati.

In [1]:
from connessione import get_db

db = get_db()
film = db["film"]

## 1. Ricerca per titolo (per prefisso)

Query: `{ "titolo": { "$regex": "^incep", "$options": "i" } }`

La regex ancorata con `^` cerca i titoli che **iniziano** con il testo digitato; l'opzione `i` ignora le maiuscole. L'ancoraggio a inizio stringa permette a MongoDB di usare l'indice `idx_titolo` invece di scorrere tutta la collezione.

`re.escape` protegge da caratteri speciali nel testo cercato (es. un titolo con le parentesi).

In [2]:
import re

def cerca_per_titolo(testo, limite=20):
    """Restituisce i film il cui titolo inizia con `testo` (maiuscole ignorate)."""
    filtro = {"titolo": {"$regex": "^" + re.escape(testo), "$options": "i"}}
    proiezione = {"titolo": 1, "anno_uscita": 1, "locandina_url": 1, "rating.tmdb_media": 1}
    return list(
        film.find(filtro, proiezione)
            .sort("titolo", 1)
            .limit(limite)
    )

In [3]:
# esempio: tutti i film che iniziano con "incep"
risultati = cerca_per_titolo("incep")
for f in risultati:
    print(f["_id"], "-", f["titolo"], f"({f.get('anno_uscita')})")

27205 - Inception (2010)


Nota sulla **proiezione**: il secondo argomento di `find` (`proiezione`) chiede solo i campi che servono alla lista dei risultati (titolo, anno, locandina, voto), invece del documento intero con cast e troupe. Una lista di ricerca resta così leggera; la scheda completa la caricheremo solo all'apertura del film.

## 2. Scheda film per id

Query: `{ "_id": 27205 }`

La lettura per chiave primaria usa l'indice automatico su `_id`. Restituisce il documento **intero** in una singola read: titolo, cast, troupe, produzione, disponibilità. È la dimostrazione della località del dato.

In [4]:
def scheda_film(id_film):
    """Restituisce il documento completo di un film dato il suo id, o None."""
    return film.find_one({"_id": id_film})

In [5]:
# esempio: scheda completa (usa un id ottenuto dalla ricerca sopra)
f = scheda_film(risultati[0]["_id"]) if risultati else None
if f:
    print("Titolo:", f["titolo"])
    print("Anno:", f.get("anno_uscita"))
    print("Generi:", f.get("generi"))
    print("Rating TMDB:", f["rating"]["tmdb_media"])
    print("Primi 3 del cast:", [c["nome"] for c in f.get("cast", [])[:3]])
    print("Disponibile in streaming:", f.get("disponibile_su_IT", {}).get("streaming"))

Titolo: Inception
Anno: 2010
Generi: ['Azione', 'Fantascienza', 'Avventura']
Rating TMDB: 8.372
Primi 3 del cast: ['Leonardo DiCaprio', 'Joseph Gordon-Levitt', 'Ken Watanabe']
Disponibile in streaming: ['Netflix', 'Timvision', 'HBO Max Amazon Channel', 'HBO Max']


## 3. Filtro per genere

Query: `{ "generi": "Azione" }` ordinata per `rating.tmdb_media` decrescente.

Sfrutta l'indice composto `idx_genere_rating`: il filtro sul genere e l'ordinamento per voto sono serviti da un solo indice. `generi` è un array, quindi il match trova i film che **contengono** quel genere.

In [6]:
def film_per_genere(genere, limite=20):
    """Film di un genere, dal più votato."""
    filtro = {"generi": genere}
    proiezione = {"titolo": 1, "anno_uscita": 1, "generi": 1, "rating.tmdb_media": 1}
    return list(
        film.find(filtro, proiezione)
            .sort("rating.tmdb_media", -1)
            .limit(limite)
    )

In [7]:
# esempio: i migliori film d'Azione
for f in film_per_genere("Azione", limite=10):
    print(f"{f['rating']['tmdb_media']:.1f}  {f['titolo']} ({f.get('anno_uscita')})")

9.3  Milky☆Subway: The Galactic Limited Express - Il film (2026)
8.6  My Dearest Assassin (2026)
8.5  Protector (2026)
8.5  Il cavaliere oscuro (2008)
8.5  Il Signore degli Anelli - Il ritorno del re (2003)
8.5  La leggenda di Hei 2 (2025)
8.5  Miraculous World : Tokyo, Stellar Force (2025)
8.5  Chainsaw Man: La storia di Reze (2025)
8.4  I sette samurai (1954)
8.4  Il Signore degli Anelli - La Compagnia dell'Anello (2001)


## Prossimi passi (per gradi)

Aggiunte naturali quando vorrai ampliare il modulo:
- filtro per **anno** o intervallo di anni (usa `idx_anno`)
- **discovery dei più votati** con soglia minima di voti (per togliere i film con pochissime valutazioni)
- **paginazione** dei risultati con `skip` e `limit`
- filtri combinati (genere + anno + voto minimo)